# Figure 2

Visual and word directions for tiger.

Inputs: `lexicon/manifest.parquet` and the word images from `scripts/stimuli/render.py`; `lexicon/concepts.parquet` from `scripts/stimuli/class_names.py`; `concepts/images.parquet` and the class images from `scripts/stimuli/class_images.py`.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import ft2font
from matplotlib.font_manager import FontProperties, get_font
from matplotlib.patches import Arc, Circle, FancyBboxPatch, PathPatch, Wedge
from matplotlib.path import Path as MPath
from matplotlib.textpath import TextPath
from matplotlib.transforms import Affine2D
from PIL import Image

from src.analysis.directions import CONCEPT_IMAGES, manifest
from src.config import local_path
from src.paper import style
from src.paper.results import class_names
from src.stimuli.typography import n_slots

style.setup_notebook()

WORD = "tiger"

#: Three of the 96 backgrounds. None shows an animal, and each sets the word in
#: large or medium type on a panel, so the word image and its control image read
#: at print size. The three differ in font, ink and polarity.
SLOTS = (36, 56, 1)

#: Six of the class's validation images, chosen for variety of pose, scale and
#: setting, none with detected text. The visual class mean averages every image
#: of the class, so the choice changes no number.
CLASS_KEYS = ("417500de3537deb5", "5c5500aacf1350a5", "58daddf44fdbd422",
              "2ac5612ef42a2dbe", "e0623ee3e00f3d96", "8149d9d58046f1ae")

MAIN = manifest()
CLASS_NAMES = class_names()
IMAGES = manifest(CONCEPT_IMAGES)
WNID = CLASS_NAMES.set_index("word").loc[WORD, "wnid"]
CLASS_ROWS = IMAGES[IMAGES.wnid.eq(WNID)].set_index("key")
N_CLASS_IMAGES = len(CLASS_ROWS)
N_SLOTS = n_slots()

## Counts in the caption

In [ ]:
print(f"ImageNet-1k validation images of {WORD}: {N_CLASS_IMAGES}")
print(f"backgrounds: {N_SLOTS}")
print(f"classes: {IMAGES.wnid.nunique():,}")
print(f"class names: {len(CLASS_NAMES)}")

## Drawing primitives

The canvas is one axes in points, so every length is a physical length on the page.

In [ ]:
#: Every formula is set in Computer Modern, as the paper's equations are.
MATH = {"math_fontfamily": "cm"}
#: The display summation sign of Computer Modern, from the font Matplotlib ships.
SUMMATION = (matplotlib.get_data_path() + "/fonts/ttf/cmex10.ttf", "summationdisplay")


def word_image(slot, condition="word"):
    rows = MAIN[MAIN.word.eq(WORD) & MAIN.slot.eq(slot) & MAIN.condition.eq(condition)]
    return np.asarray(Image.open(local_path(rows.image_path.item())).convert("RGB"))


def class_image(key):
    return np.asarray(Image.open(local_path(CLASS_ROWS.loc[key, "image_path"])).convert("RGB"))


def canvas(width_pt, height_pt):
    fig = plt.figure(figsize=(width_pt / 72, height_pt / 72))
    ax = fig.add_axes((0, 0, 1, 1))
    ax.set_xlim(0, width_pt)
    ax.set_ylim(0, height_pt)
    ax.set_aspect("equal")
    ax.set_axis_off()
    return fig, ax


def label(ax, x, y, text, *, size=style.TEXT_SIZE, color=style.INK,
          ha="center", va="center", **kw):
    return ax.text(x, y, text, fontsize=size, color=color, ha=ha, va=va, **kw)


def extent(fig, ax, artist):
    """An artist's box in data coordinates, which on this canvas are points."""
    return artist.get_window_extent(fig.canvas.get_renderer()).transformed(
        ax.transData.inverted())


def rounded(ax, x, y, w, h, face, *, r, edge="none", lw=0.6, z=1.0):
    patch = FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                           facecolor=face, edgecolor=edge, lw=lw, zorder=z)
    ax.add_patch(patch)
    return patch


def tile(ax, img, x, y, s, *, r):
    """An image in a square with slightly rounded corners, lower-left at (x, y)."""
    art = ax.imshow(img, extent=(x, x + s, y, y + s), zorder=2.0,
                    interpolation="lanczos")
    art.set_clip_path(rounded(ax, x, y, s, s, "none", r=r, z=2.0))
    rounded(ax, x, y, s, s, "none", r=r, edge=style.RULE, lw=0.4, z=2.1)


def minus(ax, x, y, *, w=None, lw=None):
    """A minus on the math axis, by default as long and heavy as a bar of "="."""
    w = MINUS_W if w is None else w
    ax.plot([x - w / 2, x + w / 2], [y, y], color=style.INK,
            lw=MINUS_LW if lw is None else lw, solid_capstyle="butt", zorder=4)


def bracket(ax, x, y0, y1, *, facing, arm, lw):
    """A square bracket from y0 to y1 whose ink ends at x, y0 and y1.

    ``facing`` is +1 for an opening bracket, whose arms point right.
    """
    spine, top, bottom = x + facing * lw / 2, y1 - lw / 2, y0 + lw / 2
    ax.plot([x + facing * arm, spine, spine, x + facing * arm],
            [top, top, bottom, bottom], color=style.INK, lw=lw,
            solid_joinstyle="miter", solid_capstyle="butt", zorder=3)


def edges_behind(ax, x, y, w, h, *, n, step, r):
    """``n`` outlined card edges to the right of a card at (x, y), for the terms not drawn."""
    for k in range(n, 0, -1):
        rounded(ax, x + k * step, y, w, h, style.WHITE, r=r, edge=style.RULE, lw=0.4,
                z=0.9 - 0.05 * k)


def ink(text, size):
    """The ink box of ``text`` at ``size``, from an origin on its baseline."""
    prop = FontProperties(family="Noto Sans", size=size, **MATH)
    return TextPath((0, 0), text, size=size, prop=prop).get_extents()


def equation_text(ax, x, y, text, size):
    """Math on a baseline at (x, y)."""
    return label(ax, x, y, text, size=size, ha="left", va="baseline", **MATH)


def summation_sign(height):
    """The summation sign as an outline ``height`` tall, ink from x = 0 and centered on y = 0.

    An outline keeps the sign a vector shape and embeds no further font in the PDF.
    """
    font = get_font(SUMMATION[0])
    font.set_size(100, 72)
    font.load_glyph(font.get_name_index(SUMMATION[1]),
                    flags=ft2font.LoadFlags.NO_HINTING)
    verts, codes = font.get_path()
    path = MPath(verts, codes)
    box = path.get_extents()
    return (Affine2D().translate(-box.x0, -(box.y0 + box.y1) / 2)
            .scale(height / box.height).transform_path(path))


def sum_operator(ax, x, axis, count, index):
    """``1/count`` and a summation sign with ``index`` beneath it, from x rightward.

    The fraction rule and the sign's center lie on the math axis. The index
    hangs INDEX_DROP below the sign, or lower if its ink would come closer than
    INDEX_GAP.
    """
    numerator, denominator = "$1$", f"${count}$"
    one = ink(numerator, FRACTION_SIZE)
    den = ink(denominator, FRACTION_SIZE)
    rule = max(one.x1, den.x1) + 2 * FRACTION_OVERHANG
    center = x + rule / 2
    equation_text(ax, center - (one.x0 + one.x1) / 2,
                  axis + FRACTION_RULE / 2 + FRACTION_GAP - one.y0, numerator,
                  FRACTION_SIZE)
    equation_text(ax, center - (den.x0 + den.x1) / 2,
                  axis - FRACTION_RULE / 2 - FRACTION_GAP - den.y1, denominator,
                  FRACTION_SIZE)
    ax.plot([x, x + rule], [axis, axis], color=style.INK, lw=FRACTION_RULE,
            solid_capstyle="butt")
    sign = Affine2D().translate(x + rule + OPERATOR_GAP, axis).transform_path(
        summation_sign(OPERATOR_HEIGHT))
    ax.add_patch(PathPatch(sign, facecolor=style.INK, edgecolor="none", zorder=3))
    box = sign.get_extents()
    mark = ink(rf"${index}$", INDEX_SIZE)
    equation_text(ax, (box.x0 + box.x1) / 2 - (mark.x0 + mark.x1) / 2,
                  box.y0 - max(INDEX_DROP, INDEX_GAP + mark.y1), rf"${index}$",
                  INDEX_SIZE)

## Geometry

The six class images sit over the three word images and their control images on one column grid.

In [ ]:
WIDTH_PT = style.TEXT_WIDTH_IN * 72
TILE = 36.0
TILE_RADIUS = 0.8
CARD_RADIUS = 1.6
CARD_PAD = 2.5
#: The gap between a pair's two tiles equals the gap between pairs, card
#: padding included, so the class images fall on an even rhythm.
PAIR_GAP = 4.0
MINUS_ROOM = 2 * CARD_PAD + PAIR_GAP
CARD_FILL = "#E6EAE8"
PAIR_W = 2 * TILE + MINUS_ROOM + 2 * CARD_PAD
BACK_EDGES = 3
BACK_STEP = 2.2

EQUATION_SIZE = 8.0
SMALL = style.TEXT_SIZE
#: The math axis: the center of "=" above the baseline. The fraction rule, the
#: summation sign and every minus sit on it, and it runs through the tiles' center.
AXIS = (ink("$=$", EQUATION_SIZE).y0 + ink("$=$", EQUATION_SIZE).y1) / 2
REL_GAP = 2.4            # left-hand side's ink to the ink of "="
FRACTION_SPACE = 2.8     # ink of "=" to the fraction rule
FRACTION_SIZE = EQUATION_SIZE
FRACTION_RULE = 0.04 * EQUATION_SIZE  # TeX's default rule
FRACTION_GAP = 1.4       # rule to numeral ink, above and below
FRACTION_OVERHANG = 0.5  # rule past the wider numeral's ink, each side
OPERATOR_HEIGHT = 12.0   # the sign's ink height
OPERATOR_GAP = 2.6       # fraction rule to the sign's ink
INDEX_SIZE = SMALL
INDEX_DROP = 5.8         # sign's lowest ink to the index's baseline, at least
INDEX_GAP = 1.9          # sign's lowest ink to the index's highest ink, at least
#: The minus in each pair card is as long as one bar of "=" and as heavy as
#: TeX's default rule, 0.04 em.
MINUS_W = ink("$=$", EQUATION_SIZE).width
MINUS_LW = 0.04 * EQUATION_SIZE

HEAD = 13.0          # room for a column head above a line
#: A column head is the paper's notation for the term beneath it, one step
#: below the equations.
HEAD_SIZE = 7.25
HEAD_RISE = 3.8      # head baseline above the top of the card beneath it
BOTTOM = 6.0
TOP_ROOM = 0.3
BRACKET_ARM = 2.6
BRACKET_LW = 0.7         # the stem of the figure's 8 pt text
BRACKET_OVERSHOOT = 2.0  # past the card's edge, above and below

#: Where the ink of every "=" ends, set so the ink margins balance.
X_EQUALS = 26.25
BRACKET_SPACE = 3.0      # the summation sign's ink to the opening bracket


def operator_width(count):
    """From the ink of "=" to the ink end of the summation sign, for 1/count."""
    one = ink("$1$", FRACTION_SIZE)
    den = ink(f"${count}$", FRACTION_SIZE)
    rule = max(one.x1, den.x1) + 2 * FRACTION_OVERHANG
    return (FRACTION_SPACE + rule + OPERATOR_GAP
            + summation_sign(OPERATOR_HEIGHT).get_extents().width)


#: The opening bracket follows the wider of the two operators, and the first
#: card follows the bracket.
X_OPEN = X_EQUALS + max(operator_width(N_CLASS_IMAGES), operator_width(N_SLOTS)) \
    + BRACKET_SPACE
X_CARDS = X_OPEN + 4.0

#: The inset is schematic: the two directions from a common origin, each ending
#: level with its line. Lengths and the angle are for layout and measure nothing.
ORIGIN_GAP = 9.0         # closing brackets to the origin
GUIDE_GAP = 3.0          # closing brackets to the start of each guide
RAY_REACH = 29.0
ANGLE_RADIUS = 10.0
VECTOR_SIZE = 7.25
GUIDE_WIDTH = 0.48
GUIDE_DASH = (0, (1.8, 2.2))

COLUMNS = []
for k in range(len(SLOTS)):
    left = X_CARDS + k * (PAIR_W + PAIR_GAP)
    COLUMNS += [left + CARD_PAD, left + CARD_PAD + TILE + MINUS_ROOM]
X_LAST = X_CARDS + len(SLOTS) * PAIR_W + (len(SLOTS) - 1) * PAIR_GAP
X_CLOSE = X_LAST + BACK_EDGES * BACK_STEP + 5.5

#: Tile bottoms.
Y_WORD = BOTTOM + CARD_PAD
Y_CLASS = Y_WORD + TILE + CARD_PAD + HEAD + CARD_PAD
HEIGHT_PT = Y_CLASS + TILE + CARD_PAD + HEAD + TOP_ROOM

## The figure

The left-hand sides are the uncentered visual class mean and word difference, and the inset shows the visual direction and the word direction.

In [ ]:
CLASS_HEADS = (r"$f(x_i)$",)
WORD_HEADS = (r"$f(x_s \oplus w)$", r"$f(x_s \oplus \tilde{w}_s)$")


def column_heads(fig, ax, y, heads):
    """One head per column from the left, its ink centered over its tile, on the baseline y."""
    for x, head in zip(COLUMNS, heads):
        text = label(ax, 0.0, y, head, size=HEAD_SIZE, ha="left", va="baseline", **MATH)
        box = extent(fig, ax, text)
        text.set_x(x + TILE / 2 - (box.x0 + box.x1) / 2)


def left_side(ax, y, lhs, count, index):
    """``lhs =`` and the operator, on a math axis through the tiles' center."""
    axis = y + TILE / 2
    baseline = axis - AXIS
    equals = ink("$=$", EQUATION_SIZE)
    side = ink(lhs, EQUATION_SIZE)
    x = X_EQUALS - equals.x1
    equation_text(ax, x, baseline, "$=$", EQUATION_SIZE)
    equation_text(ax, x + equals.x0 - REL_GAP - side.x1, baseline, lhs, EQUATION_SIZE)
    sum_operator(ax, X_EQUALS + FRACTION_SPACE, axis, count, index)


def brackets(ax, y):
    for x, facing in ((X_OPEN, 1), (X_CLOSE, -1)):
        bracket(ax, x, y - CARD_PAD - BRACKET_OVERSHOOT,
                y + TILE + CARD_PAD + BRACKET_OVERSHOOT, facing=facing,
                arm=BRACKET_ARM, lw=BRACKET_LW)


def draw_class_line(fig, ax):
    """The visual class mean over the class's validation images."""
    for k, key in enumerate(CLASS_KEYS):
        if k == len(CLASS_KEYS) - 1:
            edges_behind(ax, COLUMNS[k], Y_CLASS, TILE, TILE,
                         n=BACK_EDGES, step=BACK_STEP, r=TILE_RADIUS)
        tile(ax, class_image(key), COLUMNS[k], Y_CLASS, TILE, r=TILE_RADIUS)
    brackets(ax, Y_CLASS)
    left_side(ax, Y_CLASS, r"$\hat{\nu}_{\mathrm{tiger}}$", N_CLASS_IMAGES, "i")
    column_heads(fig, ax, Y_CLASS + TILE + CARD_PAD + HEAD_RISE, CLASS_HEADS)


def draw_word_line(fig, ax):
    """The word difference over the backgrounds: word image minus control image."""
    y = Y_WORD
    for k, slot in enumerate(SLOTS):
        left = X_CARDS + k * (PAIR_W + PAIR_GAP)
        if k == len(SLOTS) - 1:
            edges_behind(ax, left, y - CARD_PAD, PAIR_W, TILE + 2 * CARD_PAD,
                         n=BACK_EDGES, step=BACK_STEP, r=CARD_RADIUS)
        rounded(ax, left, y - CARD_PAD, PAIR_W, TILE + 2 * CARD_PAD, CARD_FILL,
                r=CARD_RADIUS)
        word_x, control_x = COLUMNS[2 * k], COLUMNS[2 * k + 1]
        tile(ax, word_image(slot), word_x, y, TILE, r=TILE_RADIUS)
        minus(ax, word_x + TILE + MINUS_ROOM / 2, y + TILE / 2)
        tile(ax, word_image(slot, "scramble"), control_x, y, TILE, r=TILE_RADIUS)
    brackets(ax, y)
    left_side(ax, y, r"$\hat{\tau}_{\mathrm{tiger}}$", N_SLOTS, "s")
    base = y + TILE + CARD_PAD + HEAD_RISE
    column_heads(fig, ax, base, WORD_HEADS)
    head_axis = ink("$=$", HEAD_SIZE)
    minus(ax, COLUMNS[0] + TILE + MINUS_ROOM / 2, base + (head_axis.y0 + head_axis.y1) / 2,
          w=head_axis.width, lw=0.04 * HEAD_SIZE)


def draw_inset(ax):
    """The visual direction and the word direction, and the cosine similarity between them.

    The origin sits midway between the lines, and each direction ends level
    with its line, joined to it by a dotted guide.
    """
    ox = X_CLOSE + ORIGIN_GAP
    oy = (Y_WORD - CARD_PAD + Y_CLASS + TILE + CARD_PAD) / 2
    levels = {"nu": Y_CLASS + TILE / 2, "tau": Y_WORD + TILE / 2}
    angle = {key: np.degrees(np.arctan2(y - oy, RAY_REACH)) for key, y in levels.items()}
    for y in levels.values():
        ax.plot([X_CLOSE + GUIDE_GAP, ox + RAY_REACH], [y, y],
                color=style.RULE, lw=GUIDE_WIDTH, ls=GUIDE_DASH, zorder=0.5)
    ax.add_patch(Wedge((ox, oy), ANGLE_RADIUS, angle["tau"], angle["nu"],
                       facecolor=style.SEMANTIC, alpha=0.12, edgecolor="none", zorder=2))
    ax.add_patch(Arc((ox, oy), 2 * ANGLE_RADIUS, 2 * ANGLE_RADIUS, theta1=angle["tau"],
                     theta2=angle["nu"], color=style.SEMANTIC, lw=0.9, zorder=4))
    label(ax, ox + ANGLE_RADIUS + 4.0, oy, "Cosine\nsimilarity", size=SMALL, ha="left",
          va="center", linespacing=1.0, color=style.SEMANTIC)
    names = {"nu": r"$\nu_{\mathrm{tiger}}$", "tau": r"$\tau_{\mathrm{tiger}}$"}
    for key, text in names.items():
        tip = (ox + RAY_REACH, levels[key])
        ax.plot([ox, tip[0]], [oy, tip[1]], color=style.REFERENCE, lw=0.72,
                solid_capstyle="round", zorder=3)
        ax.add_patch(Circle(tip, 1.66, facecolor=style.WHITE, edgecolor=style.REFERENCE,
                            lw=0.7, zorder=5))
        label(ax, tip[0] + 3.2, tip[1], text, size=VECTOR_SIZE, ha="left", **MATH)
    ax.add_patch(Circle((ox, oy), 1.1, facecolor=style.CONTROL, edgecolor="none", zorder=5))


def draw_figure():
    fig, ax = canvas(WIDTH_PT, HEIGHT_PT)
    draw_class_line(fig, ax)
    draw_word_line(fig, ax)
    draw_inset(ax)
    return fig

In [ ]:
fig = draw_figure()
# The PDF is vector except the images. A tile is half an inch wide, so at 600 dpi
# it holds 300 pixels, more than the 256 of each image; the style's 300 dpi gives 150.
style.save(fig, "figure2_datasets", dpi=600)
plt.show()